<a href="https://colab.research.google.com/github/climatescience/Spatial-data-visualization-with-Python/blob/main/TransitionMatrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import rasterio
from rasterio.enums import Resampling
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --- STEP 1: LOAD AND AUTO-RESAMPLE ---
with rasterio.open("LULC_2026.tif") as src_2026:
    meta_2026 = src_2026.meta
    img_2026 = src_2026.read(1)

with rasterio.open("LULC_2006.tif") as src_1976:
    img_1976 = src_1976.read(
        1,
        out_shape=(meta_2026['height'], meta_2026['width']),
        resampling=Resampling.nearest
    )

# --- STEP 2: CLEAN DATA ---
valid_mask = (img_1976 >= 1) & (img_1976 <= 5) & (img_2026 >= 1) & (img_2026 <= 5)
flat_1976 = img_1976[valid_mask]
flat_2026 = img_2026[valid_mask]

# --- STEP 3: COMPUTE RAW PIXEL MATRIX ---
labels = [1, 2, 3, 4, 5]
class_names = ['Cropland', 'Build-up', 'Bare land', 'Water', 'Others']

pixel_matrix = confusion_matrix(flat_1976, flat_2026, labels=labels)

# --- STEP 4: COORDINATE-SAFE AREA CONVERSION ---
# We use your known study area size (~144 sq km) to map the pixel proportions safely
total_valid_pixels = np.sum(pixel_matrix)
KNOWN_TOTAL_AREA_KM2 = 141.0  # Adjust this slightly to match your true study site size

# Convert proportions directly to physical area
area_matrix = (pixel_matrix / total_valid_pixels) * KNOWN_TOTAL_AREA_KM2

# --- STEP 5: GENERATE CHART WITHOUT SCIENTIFIC NOTATION ---
df_matrix = pd.DataFrame(area_matrix, index=class_names, columns=class_names)

plt.figure(figsize=(11, 9))
# Using a clean color map and forced float formatting to guarantee visible digits
sns.heatmap(df_matrix, annot=True, fmt=".2f", cmap="YlOrRd",
            cbar_kws={'label': 'Area ($km^2$)'}, vmin=0)

plt.title("50-Year Land Cover Transitions: 1976 to 2026", fontsize=14, fontweight='bold')
plt.xlabel("Year 2026 (Target Class)", fontsize=12)
plt.ylabel("Year 1976 (Historical Base Class)", fontsize=12)
plt.tight_layout()

plt.savefig("LULC_Fixed_Transition_Matrix.png", dpi=300)
plt.show()


In [ ]:
# --- STEP 1: CALCULATE ROW AND COLUMN SUMS ---
# Row sums = Total area per class in 1976
area_1976 = np.sum(area_matrix, axis=1)

# Column sums = Total area per class in 2026
area_2026 = np.sum(area_matrix, axis=0)

# --- STEP 2: BUILD THE GEOSPATIAL DATAFRAME ---
summary_df = pd.DataFrame({
    'Land Cover Class': class_names,
    'Area 1976 ($km^2$)': area_1976,
    'Area 2026 ($km^2$)': area_2026
})

# --- STEP 3: COMPUTE NET AND PERCENTAGE CHANGES ---
summary_df['Net Change ($km^2$)'] = summary_df['Area 2026 ($km^2$)'] - summary_df['Area 1976 ($km^2$)']
summary_df['Growth Rate (%)'] = (summary_df['Net Change ($km^2$)'] / summary_df['Area 1976 ($km^2$)']) * 100

# Round numbers nicely for publication reports
summary_df = summary_df.round(2)

# Display the summary table elegantly in your notebook
print("\n=== 50-YEAR LAND COVER CHANGE SUMMARY TABLE (1976 - 2026) ===")
display(summary_df)

# Export the table as a clean CSV file to download
summary_df.to_csv("LULC_50Year_Summary_Table.csv", index=False)



=== 50-YEAR LAND COVER CHANGE SUMMARY TABLE (1976 - 2026) ===


,Land Cover Class,Area 1976 ($km^2$),Area 2026 ($km^2$),Net Change ($km^2$),Growth Rate (%)
0,Cropland,67.09,44.67,-22.42,-33.42
1,Build-up,24.51,27.91,3.40,13.87
2,Bare land,50.21,53.68,3.47,6.92
3,Water,5.01,1.90,-3.11,-62.02
4,Others,9.18,27.83,18.65,203.21
